# Feature Cleaning Pipeline

Prepares the merged feature file for model training.

**Cleaning steps applied:**
1. Remove unnamed columns
2. Separate centroid/perimeter NaNs (absence) from coverage NaNs (structural zeros)
3. Fill coverage and grid NaNs with 0 (structural zeros — no detection means zero coverage)
4. Remove invalid all-zero rows (failed Street View requests)
5. Remove exact duplicate rows
6. Remove known sky-like duplicates (2285_* / 2790_*)
7. Normalise object area features (divide by 640x640 = 409,600 pixels)
8. Remove zero-variance columns
9. Remove near-zero columns (< 0.1% non-zero)

**Boss observations addressed:**
- Obs 1: Location-level CV splitting — point_id extracted from image_name for GroupKFold
- Obs 2: Object area normalisation — divided by total image pixels (409,600)
- Obs 3: Centroid zero vs absence — cx, cy, perimeter keep NaN when class not detected

**Output:** cleaned CSV ready for modelling, plus a location ID mapping file for CV splitting.

## 1. Imports

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# image resolution — used for area normalisation
IMAGE_PIXELS = 640 * 640  # 409,600

print('Imports loaded.')

## 2. Configuration

In [ ]:
# ============================================================
# PATHS
# ============================================================
FEATURES_PATH = r'C:/Users/researcher/Documents/2. Job/1. tasks/14. attempt/features/processed_features_merged_02_06_26.csv'
CLEANED_PATH  = r'C:/Users/researcher/Documents/2. Job/1. tasks/14. attempt/features/processed_features_merged_02_06_26_cleaned_v2.csv'
LOCID_PATH    = r'C:/Users/researcher/Documents/2. Job/1. tasks/14. attempt/features/image_location_ids.csv'

# ============================================================
# NEAR-ZERO THRESHOLD
# columns non-zero in fewer than this fraction of images are removed
# ============================================================
NEAR_ZERO_THRESHOLD = 0.001  # 0.1%

print('Configuration loaded.')
print(f'  Input  : {FEATURES_PATH}')
print(f'  Output : {CLEANED_PATH}')
print(f'  Loc IDs: {LOCID_PATH}')

## 3. Load raw features

In [ ]:
print('Loading raw features...')
raw_df        = pd.read_csv(FEATURES_PATH)
initial_shape = raw_df.shape
print(f'  Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns')

# classify all feature columns by family
_grid_re = re.compile(r'_\d+_\d+$')

def classify_cols(df):
    all_cols     = [c for c in df.columns if c != 'image_name']
    cx_cols      = [c for c in all_cols if c.startswith('seg_') and c.endswith('_cx')]
    cy_cols      = [c for c in all_cols if c.startswith('seg_') and c.endswith('_cy')]
    peri_cols    = [c for c in all_cols if c.startswith('seg_') and c.endswith('_perimeter')]
    spatial_cols = cx_cols + cy_cols + peri_cols
    area_cols    = [c for c in all_cols if c.startswith('obj_') and '_area_' in c]
    other_cols   = [c for c in all_cols if c not in spatial_cols + area_cols]
    return spatial_cols, area_cols, other_cols, cx_cols, cy_cols, peri_cols

spatial_cols, area_cols, other_cols, cx_cols, cy_cols, peri_cols = classify_cols(raw_df)
print(f'  Centroid/perimeter cols : {len(spatial_cols)}')
print(f'  Object area cols        : {len(area_cols)}')
print(f'  All other cols          : {len(other_cols)}')

## 4. Extract point ID for location-level CV splitting

Each image name follows the pattern `{point_id}_{heading}.jpg`.
All four headings from the same point must stay together in the same CV fold
to prevent location leakage between training and test sets.

This mapping file is used by the modelling notebook when constructing folds.

In [ ]:
# extract point_id from image_name: '1234_90.jpg' -> '1234'
raw_df['point_id'] = raw_df['image_name'].str.extract(r'^(\d+)_')

n_images    = len(raw_df)
n_locations = raw_df['point_id'].nunique()
headings_per_loc = raw_df.groupby('point_id').size()

print(f'  Total images    : {n_images:,}')
print(f'  Unique locations: {n_locations:,}')
print(f'  Headings per location:')
print(headings_per_loc.value_counts().sort_index().to_string())

# note any locations with fewer than 4 headings
incomplete = headings_per_loc[headings_per_loc < 4]
if len(incomplete) > 0:
    print(f'\n  Locations with < 4 headings: {len(incomplete)}')
    print(incomplete.head(10).to_string())

## 5. Cleaning pipeline

**Step-by-step rationale:**

**Step 1** — Unnamed columns are artefacts of CSV export.

**Step 2** — Centroid and perimeter NaNs are kept as NaN. When a segmentation class is not detected in an image, its centroid (cx, cy) is undefined. Filling with 0 would confuse absence with detection at coordinate (0, 0), which is the top-left corner of the image. Coverage and grid NaNs are genuine structural zeros (no coverage = 0 proportion) and are filled with 0 as before.

**Step 3** — Five rows where all features are zero or NaN are confirmed failed Street View requests and are removed.

**Step 4** — Exact duplicate rows (same feature vector) are removed, keeping the first occurrence. Catches same-heading-across-point-IDs pairs.

**Step 5** — Sky-like duplicates (2285_* / 2790_*) have slightly different centroid values so exact matching does not catch them. Removed explicitly.

**Step 6** — Object area features store raw bounding box pixel counts. Divided by 409,600 (640x640) to bring values into the 0-1 range, consistent with all other features and interpretable as fraction of image area.

**Step 7** — Zero-variance columns (nunique <= 1) carry no signal.

**Step 8** — Columns non-zero in fewer than 0.1% of images are too sparse to learn reliably.

In [ ]:
df            = raw_df.copy()
initial_shape = (len(df), len([c for c in df.columns
                                if c not in ('image_name', 'point_id')]))

# re-classify on working copy
spatial_cols, area_cols, other_cols, cx_cols, cy_cols, peri_cols = classify_cols(df)

# ── Step 1: remove unnamed columns ───────────────────────────────────────────
unnamed = [c for c in df.columns
           if str(c).startswith('Unnamed') or not str(c).strip()]
df = df.drop(columns=unnamed) if unnamed else df
print(f'Step 1 — unnamed columns removed              : {len(unnamed)}')

# ── Step 2: selective NaN fill ───────────────────────────────────────────────
# centroid and perimeter: keep NaN (absence is different from position 0)
# everything else: fill with 0 (structural zero = class not detected)
non_spatial = [c for c in df.columns
               if c not in spatial_cols
               and c not in ('image_name', 'point_id')]
nan_non_spatial = int(df[non_spatial].isna().sum().sum())
nan_spatial     = int(df[spatial_cols].isna().sum().sum())
df[non_spatial] = df[non_spatial].fillna(0)
print(f'Step 2 — coverage/grid NaNs filled with 0     : {nan_non_spatial:,}')
print(f'         centroid/perimeter NaNs kept as NaN  : {nan_spatial:,}')

# ── Step 3: remove invalid all-zero / all-null rows ──────────────────────────
# a row is invalid if every non-spatial feature is 0
# (spatial cols may still be NaN legitimately)
feature_check = [c for c in non_spatial]
all_zero_mask = (df[feature_check] == 0).all(axis=1)
invalid_imgs  = df.loc[all_zero_mask, 'image_name'].tolist()
df            = df[~all_zero_mask].copy()
print(f'Step 3 — invalid all-zero rows removed        : {len(invalid_imgs)}')
for img in invalid_imgs:
    print(f'         {img}')

# ── Step 4: remove exact duplicate rows ──────────────────────────────────────
# duplicate = identical feature vector (spatial NaNs treated as equal)
feat_for_dup = [c for c in non_spatial]
dup_mask     = df[feat_for_dup].duplicated(keep='first')
removed_dups = df.loc[dup_mask, 'image_name'].tolist()
df           = df[~dup_mask].copy()
print(f'Step 4 — exact duplicate rows removed         : {len(removed_dups)}')
for img in removed_dups:
    print(f'         {img}')

# ── Step 5: remove known sky-like duplicates ─────────────────────────────────
sky_dups = [
    '2285_0.jpg',  '2285_90.jpg',  '2285_180.jpg',  '2285_270.jpg',
    '2790_0.jpg',  '2790_90.jpg',  '2790_180.jpg',  '2790_270.jpg',
]
before_sky = len(df)
df         = df[~df['image_name'].isin(sky_dups)].copy()
sky_n      = before_sky - len(df)
print(f'Step 5 — sky-like duplicates removed          : {sky_n}')

# ── Step 6: normalise object area features ────────────────────────────────────
# area cols store raw bounding box pixel counts (can reach 381,000+)
# divide by total image pixels to get fraction of image area (0 to 1)
area_cols_present = [c for c in area_cols if c in df.columns]
before_max = df[area_cols_present].max().max() if area_cols_present else 0
df[area_cols_present] = df[area_cols_present] / IMAGE_PIXELS
after_max  = df[area_cols_present].max().max() if area_cols_present else 0
print(f'Step 6 — object area normalised by {IMAGE_PIXELS:,} pixels')
print(f'         max before: {before_max:,.1f}  ->  max after: {after_max:.6f}')

# ── Step 7: remove zero-variance columns ─────────────────────────────────────
feat_cols     = [c for c in df.columns
                 if c not in ('image_name', 'point_id')]
n_unique      = df[feat_cols].nunique(dropna=False)
zero_var_cols = n_unique[n_unique <= 1].index.tolist()
df            = df.drop(columns=zero_var_cols)
print(f'Step 7 — zero-variance columns removed        : {len(zero_var_cols)}')

# ── Step 8: remove near-zero columns ─────────────────────────────────────────
# non-zero fraction computed ignoring NaN (spatial cols)
feat_cols    = [c for c in df.columns
                if c not in ('image_name', 'point_id')]
nonzero_frac = df[feat_cols].apply(lambda s: (s != 0).mean())
sparse_cols  = nonzero_frac[nonzero_frac < NEAR_ZERO_THRESHOLD].index.tolist()
df           = df.drop(columns=sparse_cols)
print(f'Step 8 — near-zero columns removed            : {len(sparse_cols)} (< {NEAR_ZERO_THRESHOLD*100:.1f}% non-zero)')

# ── Summary ───────────────────────────────────────────────────────────────────
feat_cols = [c for c in df.columns if c not in ('image_name', 'point_id')]
print()
print(f'Before cleaning : {initial_shape[0]:,} rows x {initial_shape[1]:,} features')
print(f'After cleaning  : {len(df):,} rows x {len(feat_cols):,} features')

## 6. Verify centroid NaN semantics

Confirm that cx/cy NaN correctly reflects absence of detection,
and that 0 values in cx/cy are genuine detections at the image edge,
not artefacts of the NaN fill.

In [ ]:
cx_present = [c for c in cx_cols if c in df.columns]
if cx_present:
    nan_cx   = df[cx_present].isna().sum().sum()
    zero_cx  = (df[cx_present] == 0).sum().sum()
    total_cx = len(df) * len(cx_present)

    print('Centroid NaN vs zero verification:')
    print(f'  Total centroid cells    : {total_cx:,}')
    print(f'  NaN (class absent)      : {nan_cx:,}  ({nan_cx/total_cx*100:.1f}%)')
    print(f'  Zero (detected at 0,0)  : {zero_cx:,}  ({zero_cx/total_cx*100:.2f}%)')
    print(f'  Non-zero, non-NaN       : {total_cx - nan_cx - zero_cx:,}')
    print()
    print('  Sample: seg_Car_cx')
    col = 'seg_Car_cx'
    if col in df.columns:
        print(f'    NaN (Car absent)  : {df[col].isna().sum():,}')
        print(f'    Zero              : {(df[col]==0).sum():,}')
        print(f'    Non-zero non-NaN  : {(df[col].notna() & (df[col]!=0)).sum():,}')
        print(f'    Mean (when present): {df[col].dropna().mean():.4f}')
    print()
    print('  Cross-check: cx should be NaN if and only if coverage is 0')
    seg_car = 'seg_Car'
    if seg_car in df.columns and 'seg_Car_cx' in df.columns:
        absent_coverage   = (df[seg_car] == 0).sum()
        absent_cx         = df['seg_Car_cx'].isna().sum()
        print(f'    seg_Car == 0        : {absent_coverage:,}')
        print(f'    seg_Car_cx is NaN   : {absent_cx:,}')
        match = absent_coverage == absent_cx
        print(f'    Counts match        : {"YES" if match else "NO — investigate"}')

## 7. Post-cleaning diagnostic

In [ ]:
feat_cols = [c for c in df.columns if c not in ('image_name', 'point_id')]

# reclassify on cleaned df
_grid_re2    = re.compile(r'_\d+_\d+$')
cx_c         = [c for c in feat_cols if c.startswith('seg_') and c.endswith('_cx')]
cy_c         = [c for c in feat_cols if c.startswith('seg_') and c.endswith('_cy')]
pe_c         = [c for c in feat_cols if c.startswith('seg_') and c.endswith('_perimeter')]
seg_cov_c    = [c for c in feat_cols if c.startswith('seg_')
                and not _grid_re2.search(c)
                and not c.endswith('_cx')
                and not c.endswith('_cy')
                and not c.endswith('_perimeter')]
seg_grid_c   = [c for c in feat_cols if c.startswith('seg_') and _grid_re2.search(c)]
obj_p_c      = [c for c in feat_cols if c.startswith('obj_') and _grid_re2.search(c)
                and '_count_' not in c and '_area_' not in c]
obj_cnt_c    = [c for c in feat_cols if c.startswith('obj_') and '_count_' in c]
obj_area_c   = [c for c in feat_cols if c.startswith('obj_') and '_area_' in c]
stat_c       = [c for c in feat_cols
                if not c.startswith('seg_') and not c.startswith('obj_')]

groups_clean = {
    'Seg coverage'      : seg_cov_c,
    'Seg centroid X'    : cx_c,
    'Seg centroid Y'    : cy_c,
    'Seg perimeter'     : pe_c,
    'Seg grid'          : seg_grid_c,
    'Obj probability'   : obj_p_c,
    'Obj count'         : obj_cnt_c,
    'Obj area'          : obj_area_c,
    'Image statistics'  : stat_c,
}

print('=' * 65)
print('FEATURE FAMILY OVERVIEW — after cleaning')
print('=' * 65)
print(f'  {"Group":<28} {"Count":>6}  {"% non-zero":>12}  {"NaN cols":>10}')
print('  ' + '-' * 62)
for name, cols in groups_clean.items():
    if not cols: continue
    pct_nz  = (df[cols] != 0).mean().mean() * 100
    nan_cols = sum(df[c].isna().any() for c in cols)
    print(f'  {name:<28} {len(cols):>6}  {pct_nz:>11.1f}%  {nan_cols:>10}')
print('  ' + '-' * 62)
print(f'  {"TOTAL":<28} {len(feat_cols):>6}')

print()
# key checks
nan_total  = int(df[feat_cols].isna().sum().sum())
zero_var   = [c for c in feat_cols if df[c].nunique(dropna=False) <= 1]
near_zero  = [c for c in feat_cols if (df[c]!=0).mean() < 0.001 and c not in zero_var]
dup_rows   = int(df[feat_cols].duplicated().sum())
dup_names  = int(df['image_name'].duplicated().sum())
all_zero   = int((df[feat_cols].fillna(0)==0).all(axis=1).sum())

# area range check
area_cols_clean = [c for c in obj_area_c if c in df.columns]
area_max = df[area_cols_clean].max().max() if area_cols_clean else 0

print('QUALITY CHECKS')
print('=' * 65)
print(f'  Rows                          : {len(df):,}')
print(f'  Features                      : {len(feat_cols):,}')
print(f'  NaN total (spatial only ok)   : {nan_total:,}')
print(f'  NaN in spatial (expected)     : {df[cx_c+cy_c+pe_c].isna().sum().sum():,}')
print(f'  NaN outside spatial           : {df[[c for c in feat_cols if c not in cx_c+cy_c+pe_c]].isna().sum().sum():,}   {"OK" if df[[c for c in feat_cols if c not in cx_c+cy_c+pe_c]].isna().sum().sum()==0 else "ISSUE"}')
print(f'  Zero-variance columns         : {len(zero_var)}   {"OK" if not zero_var else "ISSUE"}')
print(f'  Near-zero columns             : {len(near_zero)}   {"OK" if not near_zero else "ISSUE"}')
print(f'  Duplicate rows                : {dup_rows}   {"OK" if dup_rows==0 else "ISSUE"}')
print(f'  Duplicate image names         : {dup_names}   {"OK" if dup_names==0 else "ISSUE"}')
print(f'  All-zero rows                 : {all_zero}   {"OK" if all_zero==0 else "ISSUE"}')
print(f'  Obj area max (should be <= 1) : {area_max:.6f}   {"OK" if area_max <= 1.0 else "ISSUE"}')

## 8. Location ID distribution

Verifies that the point_id extraction worked correctly and shows how many images
remain per location after cleaning. This directly informs the GroupKFold in the
modelling notebook.

In [ ]:
loc_counts = df.groupby('point_id').size()

print('Location distribution after cleaning:')
print(f'  Total images    : {len(df):,}')
print(f'  Unique locations: {len(loc_counts):,}')
print(f'\n  Headings per location:')
print(loc_counts.value_counts().sort_index().to_string())

# any location with only 1 image after cleaning (may affect GroupKFold)
single = loc_counts[loc_counts == 1]
if len(single) > 0:
    print(f'\n  Locations with only 1 image: {len(single)}')
    print('  (These are fine for GroupKFold but worth noting)')

## 9. Save outputs

In [ ]:
# save cleaned features — includes point_id column for GroupKFold
df.to_csv(CLEANED_PATH, index=False)
print(f'Cleaned features saved : {CLEANED_PATH}')
print(f'Shape                  : {len(df):,} rows x {len(feat_cols):,} features')
print(f'Columns saved          : {list(df.columns[:5])} ... (includes point_id)')

# save location id mapping separately for convenience
loc_map = df[['image_name', 'point_id']].copy()
loc_map.to_csv(LOCID_PATH, index=False)
print(f'\nLocation ID mapping saved: {LOCID_PATH}')
print(f'Shape                     : {loc_map.shape}')

# Composite feature construction

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

CLEANED_PATH    = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26_cleaned_v2.csv'
GPT_RANK_PATH   = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\gpt_trueskill_rankings_20260621_190450.csv'
HUMAN_RANK_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\human_trueskill_rankings_20260621_190450.csv'
OUTPUT_COMPOSITES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_23_06_26.csv'

features_df = pd.read_csv(CLEANED_PATH)
gpt_df      = pd.read_csv(GPT_RANK_PATH)
human_df    = pd.read_csv(HUMAN_RANK_PATH)

print(f'Features loaded : {features_df.shape[0]:,} rows x {features_df.shape[1]:,} cols')
print(f'GPT rankings    : {len(gpt_df):,}')
print(f'Human rankings  : {len(human_df):,}')

In [ ]:
# ── COMPOSITE FEATURE CONSTRUCTION ───────────────────────────────────────────
# Based on diagnostic: composites where data shows additive signal.
# comp_cycling_infra included per boss suggestion even though signal is near-zero
# — the Pearson threshold in the prediction notebook will decide whether to keep it.

df_comp = features_df.copy()

def safe_sum(df_, cols):
    present = [c for c in cols if c in df_.columns]
    missing = [c for c in cols if c not in df_.columns]
    if missing:
        print(f'  WARNING: missing columns skipped: {missing}')
    return df_[present].fillna(0).sum(axis=1) if present else pd.Series(0, index=df_.index)

# ── coverage composites ───────────────────────────────────────────────────────

# boss suggestion — signal is near-zero per diagnostic but included for completeness
df_comp['comp_cycling_infra'] = safe_sum(df_comp, [
    'seg_Bike Lane', 'seg_Bike Rack', 'seg_Bicycle',
    'seg_Bicyclist', 'seg_Other Rider', 'seg_Wheeled Slow'
])

# boss suggestion — confirmed strong signal (avg|r|=0.21)
df_comp['comp_motor_load'] = safe_sum(df_comp, [
    'seg_Car', 'seg_Truck', 'seg_Bus',
    'seg_Motorcycle', 'seg_Other Vehicle'
])

df_comp['comp_separation'] = safe_sum(df_comp, [
    'seg_Barrier', 'seg_Guard Rail',
    'seg_Fence', 'seg_Wall',
    'seg_Tunnel', 'seg_Bridge'
])
df_comp['comp_natural_openness'] = safe_sum(df_comp, [
    'seg_Vegetation', 'seg_Terrain',
    'seg_Mountain', 'seg_Water'
])
df_comp['comp_commercial_density'] = safe_sum(df_comp, [
    'seg_Billboard', 'seg_Street Light',
    'seg_Traffic Sign (Front)', 'seg_CCTV Camera'
])
df_comp['comp_built_density'] = safe_sum(df_comp, [
    'seg_Building', 'seg_Wall', 'seg_Bridge'
])

# ── binary flags ──────────────────────────────────────────────────────────────
df_comp['comp_has_cycling_infra']      = (df_comp['comp_cycling_infra']      > 0).astype(int)
df_comp['comp_has_motor_load']         = (df_comp['comp_motor_load']         > 0).astype(int)
df_comp['comp_has_separation']         = (df_comp['comp_separation']         > 0).astype(int)
df_comp['comp_has_natural_openness']   = (df_comp['comp_natural_openness']   > 0).astype(int)
df_comp['comp_has_commercial_density'] = (df_comp['comp_commercial_density'] > 0).astype(int)
df_comp['comp_has_built_density']      = (df_comp['comp_built_density']      > 0).astype(int)

# ── ratio composites ──────────────────────────────────────────────────────────
df_comp['comp_nature_vs_built'] = (
    df_comp['comp_natural_openness'] /
    (df_comp['comp_built_density'] + 0.001)
)
df_comp['comp_terrain_vs_motor'] = (
    safe_sum(df_comp, ['seg_Terrain', 'seg_Vegetation']) /
    (df_comp['comp_motor_load'] + 0.001)
)

# ── summary ───────────────────────────────────────────────────────────────────
comp_cols = [c for c in df_comp.columns if c.startswith('comp_')]
feat_cols = [c for c in df_comp.columns if c not in ('image_name', 'point_id')]

print(f'Composite features added         : {len(comp_cols)}')
print(f'Total features (with composites) : {len(feat_cols)}')
print()
print(f'{"Composite":<35} {"Type":<8} {"% non-zero":>12} {"Mean":>8} {"Std":>8} {"Max":>10}')
print('-' * 85)
for c in comp_cols:
    s = df_comp[c].replace([np.inf, -np.inf], np.nan).fillna(0)
    pct_nz = (s > 0).mean() * 100
    if 'has_' in c:
        ctype = 'binary'
    elif 'vs' in c:
        ctype = 'ratio'
    else:
        ctype = 'coverage'
    print(f'{c:<35} {ctype:<8} {pct_nz:>11.1f}% {s.mean():>8.4f} {s.std():>8.4f} {s.max():>10.4f}')

## Correlation check

In [ ]:
# ── CORRELATION CHECK ─────────────────────────────────────────────────────────
# Compare composites vs their strongest individual constituents

gpt_comp_merged   = gpt_df.merge(df_comp, on='image_name', how='inner')
human_comp_merged = human_df.merge(df_comp, on='image_name', how='inner')

print(f'Merged GPT   : {len(gpt_comp_merged):,}')
print(f'Merged Human : {len(human_comp_merged):,}')
print()
print('COMPOSITE vs INDIVIDUAL CORRELATIONS WITH TRUESKILL MU')
print(f'{"Feature":<38} {"r_GPT":>8} {"r_Human":>10} {"avg|r|":>8}  {"Agreement"}')
print('-' * 80)

check_cols = comp_cols + [
    'seg_Building', 'seg_Terrain', 'seg_Car',
    'seg_Vegetation', 'seg_Billboard'
]

rows = []
for c in check_cols:
    if c not in gpt_comp_merged.columns:
        continue
    xg = gpt_comp_merged[c].replace([np.inf,-np.inf],np.nan).fillna(0).values
    xh = human_comp_merged[c].replace([np.inf,-np.inf],np.nan).fillna(0).values
    yg = gpt_comp_merged['gpt_mu'].values
    yh = human_comp_merged['human_mu'].values
    rg = pearsonr(xg, yg)[0] if xg.std() > 0 else 0.0
    rh = pearsonr(xh, yh)[0] if xh.std() > 0 else 0.0
    avg = (abs(rg) + abs(rh)) / 2
    rows.append((c, rg, rh, avg))

rows.sort(key=lambda x: -x[3])
for c, rg, rh, avg in rows:
    agree  = 'AGREE'    if (rg > 0) == (rh > 0) else 'DISAGREE'
    marker = ' <-- comp' if c.startswith('comp_') else '         '
    print(f'{c:<38} {rg:>8.4f} {rh:>10.4f} {avg:>8.4f}  {agree}{marker}')

In [ ]:
# ── QUALITY CHECKS ────────────────────────────────────────────────────────────
print('QUALITY CHECKS')
print('=' * 55)

# NaN / inf
n_bad = df_comp[comp_cols].replace([np.inf,-np.inf],np.nan).isna().sum().sum()
print(f'  NaN / inf in composite cols : {n_bad}   {"OK" if n_bad == 0 else "FIX NEEDED"}')

# binary cols
binary_cols = [c for c in comp_cols if 'has_' in c]
for c in binary_cols:
    vals = sorted(df_comp[c].unique().tolist())
    ok   = set(vals).issubset({0, 1})
    print(f'  {c:<38}: {vals}  {"OK" if ok else "FIX"}')

# ratio cols
ratio_cols = [c for c in comp_cols if 'vs' in c]
for c in ratio_cols:
    n_inf = np.isinf(df_comp[c]).sum()
    n_nan = df_comp[c].isna().sum()
    print(f'  {c:<38}: inf={n_inf}  nan={n_nan}  {"OK" if n_inf==0 and n_nan==0 else "FIX"}')

# counts
feat_cols = [c for c in df_comp.columns if c not in ('image_name','point_id')]
print(f'\n  Original features  : 771')
print(f'  Composite features : {len(comp_cols)}')
print(f'  Total features     : {len(feat_cols)}')
print(f'  Rows               : {len(df_comp):,}  {"OK" if len(df_comp)==19970 else "FIX"}')

# ── SAVE ──────────────────────────────────────────────────────────────────────
df_comp.to_csv(OUTPUT_COMPOSITES_PATH, index=False, encoding='utf-8-sig')
print(f'\nSaved : {OUTPUT_COMPOSITES_PATH}')
print(f'Shape : {df_comp.shape[0]:,} rows x {df_comp.shape[1]:,} columns')

# Visually verification of the step 5: whay do we have exact duplicates images?

In [ ]:
# ── STEP 5 DIAGNOSTIC — visualise exact duplicate image pairs ─────────────────
# For each group of exact duplicates found in Step 5, fetch and display
# the images side by side so we can see why they have identical features.

import io
from PIL import Image
from azure.storage.blob import BlobServiceClient
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Azure connection (same as GPT votes notebook) ────────────────────────────
AZURE_CONN_STR = (
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
BLOB_CONTAINER = 'london'
IMAGE_PREFIX   = 'training/training/Images/'

blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

def fetch_image(image_name):
    blob_data = container_client.download_blob(
        IMAGE_PREFIX + image_name, timeout=30
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')


# ── find duplicate groups from the raw features file ─────────────────────────
# reload raw and rerun the duplicate detection exactly as in the cleaning cell

df_raw = pd.read_csv(FEATURES_PATH)

# same logic as Step 5 in cleaning
_grid_re_diag = re.compile(r'_\d+_\d+$')
_all_raw_diag = [c for c in df_raw.columns if c != 'image_name']
spatial_diag  = (
    [c for c in _all_raw_diag if c.startswith('seg_') and c.endswith('_cx')] +
    [c for c in _all_raw_diag if c.startswith('seg_') and c.endswith('_cy')] +
    [c for c in _all_raw_diag if c.startswith('seg_') and c.endswith('_perimeter')]
)
non_spatial_diag = [c for c in _all_raw_diag if c not in spatial_diag]

# fill non-spatial NaNs (same as Step 3)
df_diag = df_raw.copy()
df_diag[non_spatial_diag] = df_diag[non_spatial_diag].fillna(0)

# find duplicates
dup_mask = df_diag[non_spatial_diag].duplicated(keep=False)
dup_df   = df_diag[dup_mask][['image_name'] + non_spatial_diag].copy()

# group by feature vector — assign a group ID to each duplicate group
dup_df['group_id'] = df_diag.loc[dup_mask, non_spatial_diag].apply(
    lambda row: hash(tuple(row)), axis=1
)

groups = dup_df.groupby('group_id')['image_name'].apply(list).reset_index()
groups['group_size'] = groups['image_name'].apply(len)
groups = groups[groups['group_size'] > 1].sort_values('group_size', ascending=False)

print(f'Duplicate groups found : {len(groups)}')
print(f'Total duplicate images : {dup_df["image_name"].nunique()}')
print()
for _, row in groups.iterrows():
    print(f'  Group (size {row["group_size"]}): {row["image_name"]}')

In [ ]:
# ── DISPLAY DUPLICATE GROUPS ──────────────────────────────────────────────────
# Fetch and show each duplicate group side by side
# This lets you visually confirm why they have identical features

print(f'Fetching and displaying {len(groups)} duplicate groups...')
print('(This may take a minute depending on group count)')
print()

for _, group_row in groups.iterrows():
    images    = group_row['image_name']
    n_images  = len(images)

    fig, axes = plt.subplots(1, n_images, figsize=(5 * n_images, 5))
    fig.patch.set_facecolor('white')

    if n_images == 1:
        axes = [axes]

    for ax, img_name in zip(axes, images):
        try:
            pil_img = fetch_image(img_name)
            ax.imshow(pil_img)
            # extract point_id and heading from name
            parts = img_name.replace('.jpg', '').split('_')
            point_id = parts[0]
            heading  = parts[1] if len(parts) > 1 else '?'
            ax.set_title(
                f'{img_name}\nPoint {point_id} | Heading {heading}°',
                fontsize=9, fontweight='bold'
            )
        except Exception as e:
            ax.text(0.5, 0.5, f'Could not load\n{img_name}\n{str(e)[:40]}',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=8, color='red')
            ax.set_title(img_name, fontsize=9)

        ax.axis('off')

    fig.suptitle(
        f'Duplicate group — {n_images} images with identical feature vectors',
        fontsize=10, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()
    print()

print('Done. Check the images above.')
print('Expected explanations:')
print('  - Same physical location photographed from two different GPS points')
print('  - Street view image captured at the exact same spot at different times')
print('  - Failed or blank captures that all produce zero feature vectors')